# Genixcure PacBio PLA_6_13 — Short-read compatible result delivery

**Status:** Circular-start normalization & exact-count source **already completed** (5,140,518 BAM reads; 4,947,363 normalized mapped reads; 4,195,072 exact unique sequences).

**Objective:** Export the entire result as **`sequence,read_count,length_bp`**, precisely matching the 11 Short-read tables. No realignment or clustering occurs.

**Outputs:**
- `PLA_6_13_circular_normalized_unique_sequences.csv.gz` — full 3-column unique table (~GB, 4.2M rows)
- `PLA_6_13_circular_normalized_top100_shortread.csv` — inspectable top 100
- `PLA_6_13_shortread_format_validation.json` — count, length, SHA256, MD5 checks

**Note:** 168,456 mapped reads without a uniquely identifiable anchor, plus 24,699 unmapped reads, are not included in the exact normalized table and are reported separately. The reference-based rotation is computational only; original DNA was EcoRV-linearized.

**Run:** Run cells in order. The 8.4-GB source is read sequentially from MyDrive; do not try to open it in Excel, which has a 1,048,576-row limit. Google Drive upload uses resumable uploads.


In [ ]:
# 1. Connect to Drive and locate the authoritative completed dataset.
import os, sys, subprocess, json, hashlib, shutil, time
from pathlib import Path
from google.colab import drive, auth
import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from googleapiclient.errors import HttpError

drive.mount('/content/drive')
auth.authenticate_user()
creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
api = build('drive','v3',credentials=creds,cache_discovery=False)
ROOT=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/PacBio_PLA_6_13')
REF=ROOT/'Reference_EcoRV_Linear'
SRC_DIR=REF/'Alignment_Circular_Normalized_Count_v1'
SOURCE=SRC_DIR/'mapped_circular_start_normalized_count.csv.gz'
SUMMARY=SRC_DIR/'circular_start_normalized_summary.json'
SCRIPT=REF/'convert_pacbio_to_shortread_table.py'
DEST_DRIVE_FOLDER_ID='1VN66NJuR5EiY5-7LEsFhu9yaqWoU8G2A'
WORK=Path('/content/genixcure_pacbio_shortread_delivery')
WORK.mkdir(parents=True, exist_ok=True)
assert SOURCE.is_file() and SUMMARY.is_file() and SCRIPT.is_file(), 'Missing source/summary/converter on mounted MyDrive'
assert SOURCE.stat().st_size == 8402255312, f'Unexpected source bytes {SOURCE.stat().st_size}'
summary=json.loads(SUMMARY.read_text())
assert summary['status']=='CIRCULAR_START_NORMALIZED_COMPLETE'
assert summary['normalized_mapped_reads']==4947363
assert summary['distinct_circular_normalized_sequences']==4195072
free=shutil.disk_usage(WORK).free
assert free >= 10000000000, f'Colab local disk needs >=10GB free, found {free/1e9:.2f}GB'
print('Source confirmed:',SOURCE.name, round(SOURCE.stat().st_size/1e9,2),'GB')
print('Local free:',round(free/1e9,2),'GB')
print('Exact count target:', summary['normalized_mapped_reads'], 'reads /', summary['distinct_circular_normalized_sequences'], 'unique')
print('Delivery folder: https://drive.google.com/drive/folders/'+DEST_DRIVE_FOLDER_ID)


In [ ]:
# 2. Convert the entire 8.4GB source to the 3-column Short-read style, with continuous progress.
# Local validated output can be reused if only the Drive upload needs retrying.
TARGET=WORK/'PLA_6_13_circular_normalized_unique_sequences.csv.gz'
VALIDATION=WORK/'PLA_6_13_shortread_format_validation.json'
reuse=False
if TARGET.exists() and VALIDATION.exists():
    prev=json.loads(VALIDATION.read_text())
    reuse=(prev.get('status')=='PASS' and prev.get('unique_sequences')==4195072 and
           prev.get('read_count_sum')==4947363 and prev.get('output_file_bytes')==TARGET.stat().st_size)
if reuse:
    print('Reusing validated existing conversion (no 8.4GB re-processing).')
else:
    cmd=[sys.executable,'-u',str(SCRIPT),'--source',str(SOURCE),'--summary',str(SUMMARY),
         '--output-dir',str(WORK),'--expected-source-bytes','8402255312','--progress-every','100000']
    print('Starting full conversion (streaming; progress after every 100,000 unique sequences).',flush=True)
    with subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as proc:
        for line in proc.stdout: print(line,end='',flush=True)
        rc=proc.wait()
    if rc: raise RuntimeError(f'Conversion failed with return code {rc}. A partial file is not the final deliverable.')
result=json.loads(VALIDATION.read_text())
assert result['read_count_sum']==4947363 and result['unique_sequences']==4195072
print('Conversion PASS:',result['output_file'],'bytes:',result['output_file_bytes'])


In [ ]:
# 3. Upload the actual full file and QC outputs to the dedicated Drive delivery folder.
# Upload uses Google Drive resumable sessions. The existing file is reused when MD5 and size match.
def md5_file(p):
    h=hashlib.md5()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
    return h.hexdigest()

def send_to_drive(p):
    p=Path(p)
    local_md5=result['output_md5'] if p.name==result['output_file'] else md5_file(p)
    q=f"'{DEST_DRIVE_FOLDER_ID}' in parents and name = '{p.name}' and trashed = false"
    matches=api.files().list(q=q,fields='files(id,name,size,md5Checksum),nextPageToken',pageSize=50).execute().get('files',[])
    for f in matches:
        if int(f.get('size',-1))==p.stat().st_size and f.get('md5Checksum')==local_md5:
            print('[REUSE]',p.name, 'https://drive.google.com/file/d/'+f['id']+'/view')
            return f['id']
    media=MediaFileUpload(str(p),mimetype='application/gzip' if p.suffix=='.gz' else 'application/octet-stream',chunksize=32*1024*1024,resumable=True)
    if matches:
        target=matches[0]
        request=api.files().update(fileId=target['id'],media_body=media,fields='id,name,size,md5Checksum')
    else:
        request=api.files().create(body={'name':p.name,'parents':[DEST_DRIVE_FOLDER_ID]},media_body=media,fields='id,name,size,md5Checksum')
    response=None
    last_pct=-1
    while response is None:
        for attempt in range(6):
            try:
                status,response=request.next_chunk()
                break
            except (HttpError, OSError, ConnectionError) as exc:
                if attempt==5: raise
                pause=min(2**attempt,30)
                print(f'[UPLOAD RETRY] {p.name}: {exc} after {pause}s',flush=True)
                time.sleep(pause)
        if status is not None:
            pct=int(100*status.progress())
            if pct>=last_pct+5:
                print(f'[UPLOADING] {p.name} {pct}%',flush=True)
                last_pct=pct
    remote=api.files().get(fileId=response['id'],fields='id,name,size,md5Checksum').execute()
    if int(remote['size'])!=p.stat().st_size or remote.get('md5Checksum')!=local_md5:
        raise RuntimeError('Drive upload MD5 or size verification failed: '+p.name)
    print('[UPLOAD PASS]',p.name,'https://drive.google.com/file/d/'+remote['id']+'/view',flush=True)
    return remote['id']

files=[WORK/result['output_file'],
       WORK/'PLA_6_13_circular_normalized_top100_shortread.csv',
       WORK/'PLA_6_13_shortread_format_validation.json']
ids={p.name:send_to_drive(p) for p in files}
print('All uploads verified. Delivery folder: https://drive.google.com/drive/folders/'+DEST_DRIVE_FOLDER_ID)
print(json.dumps(ids,indent=2))


In [ ]:
# 4. Review final summary and top 20 sequences (short-read schema).
import csv
from IPython.display import display
import pandas as pd
v=json.loads((WORK/'PLA_6_13_shortread_format_validation.json').read_text())
print(json.dumps({k:v[k] for k in ['sample','status','unique_sequences','read_count_sum','full_span_read_count_sum_original','unresolved_mapped_reads_excluded','unmapped_reads_excluded','output_file','output_sha256']},indent=2))
p=WORK/'PLA_6_13_circular_normalized_top100_shortread.csv'
display(pd.read_csv(p).head(20).assign(sequence=lambda d:d['sequence'].str[:80]+'...'))
